In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
import json
from PIL import Image
from tqdm import tqdm 

# --- CẬP NHẬT ĐƯỜNG DẪN NẾU CẦN ---
IMAGE_DIR = "/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/images" 
TRAIN_JSON_PATH = "/kaggle/input/datasets/spixalo/trafic/traffic-signs-json/train.json"
VAL_JSON_PATH = "/kaggle/input/datasets/spixalo/trafic/traffic-signs-json/val.json"

def process_vqa_dataset(json_path, image_dir, split_name="Dataset"):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    formatted_data = []
    
    for item in tqdm(data, desc=f"Đang xử lý {split_name}"):
        img_name = item["image_name"]
        img_path = f"{image_dir}/{img_name}"
        question = item["question"]
        answer = item["answer"][0] 
        
        try:
            img = Image.open(img_path).convert("RGB")
            
            conversation = [
                { "role": "user",
                  "content" : [
                    {"type" : "image", "image" : img},
                    {"type" : "text",  "text"  : question} ]
                },
                { "role" : "assistant",
                  "content" : [
                    {"type" : "text",  "text"  : answer} ]
                },
            ]
            formatted_data.append({ "messages" : conversation })
            
        except Exception as e:
            tqdm.write(f"Lỗi xử lý ảnh {img_path}: {e}")
            continue
            
    # SỬA Ở ĐÂY: Trả về trực tiếp list Python, không dùng Dataset.from_list nữa
    return formatted_data

# Tạo dataset
train_dataset = process_vqa_dataset(TRAIN_JSON_PATH, IMAGE_DIR, "Tập Train")
print(f"Số lượng mẫu Train hợp lệ: {len(train_dataset)}\n")

val_dataset = process_vqa_dataset(VAL_JSON_PATH, IMAGE_DIR, "Tập Validation")
print(f"Số lượng mẫu Val hợp lệ: {len(val_dataset)}\n")

Đang xử lý Tập Train: 100%|██████████| 2075/2075 [00:21<00:00, 95.85it/s] 


Số lượng mẫu Train hợp lệ: 2075



Đang xử lý Tập Validation: 100%|██████████| 273/273 [00:02<00:00, 92.33it/s]

Số lượng mẫu Val hợp lệ: 273



In [3]:
from unsloth import FastVisionModel
import torch

# Tải mô hình Qwen2-VL 7B bản 4-bit (chống OOM)
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth", 
)

# Cấu hình LoRA (Chỉ huấn luyện một phần nhỏ tham số)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    
    r = 16,           # Rank càng cao độ chính xác càng tăng nhưng tốn RAM hơn
    lora_alpha = 16,  
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-08 05:54:32.514602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778219672.903865      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778219673.011202      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778219674.002567      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778219674.002606      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778219674.002609      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen2VisionTransformerPretrainedModel; please override in the subclass.. Falling back to pre-forward hook.


In [4]:
import numpy as np
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# 1. Định nghĩa hàm tiền xử lý logits (Tránh bị tràn RAM - OOM)
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Với mô hình Causal LM (như Qwen), dự đoán token tiếp theo sẽ bị lệch (shift) 1 bước so với nhãn.
    # Ta dùng argmax để chuyển xác suất thành ID của token dự đoán.
    preds = logits.argmax(dim=-1)[..., :-1] 
    return preds

# 2. Định nghĩa hàm tính VQA Accuracy (Exact Match)
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # Cắt bỏ token đầu tiên của labels để khớp với preds đã shift ở trên
    labels = labels[..., 1:]
    
    exact_matches = 0
    total = len(preds)
    
    for i in range(total):
        # Chỉ lấy các token thuộc câu trả lời (Hugging Face đánh dấu các token prompt bằng -100)
        valid_mask = labels[i] != -100
        pred_tokens = preds[i][valid_mask]
        label_tokens = labels[i][valid_mask]
        
        # Giải mã ID thành chuỗi văn bản, loại bỏ khoảng trắng thừa và in thường
        pred_str = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip().lower()
        label_str = tokenizer.decode(label_tokens, skip_special_tokens=True).strip().lower()
        
        # Tính điểm Exact Match
        if pred_str == label_str:
            exact_matches += 1
            
    return {"vqa_accuracy": exact_matches / total}

# Kích hoạt chế độ huấn luyện
FastVisionModel.for_training(model) 

# Cấu hình tham số huấn luyện
training_args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4, # Tổng effective batch = 8
    warmup_steps = 5,
    num_train_epochs = 5,         
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    
    output_dir = "/kaggle/working/checkpoints_2b",
    eval_strategy = "steps",            
    eval_steps = 50,                    
    save_strategy = "steps",            
    save_steps = 50,                    
    load_best_model_at_end = True,      
    
    # --- CÁC ĐIỀU CHỈNH CHO METRIC MỚI ---
    metric_for_best_model = "vqa_accuracy", # Theo dõi VQA Accuracy
    greater_is_better = True,               # Độ chính xác thì càng cao càng tốt
    
    save_total_limit = 1,               
    remove_unused_columns = False,
    dataset_text_field = "",
    dataset_kwargs = {"skip_prepare_dataset": True},
    max_length = 2048,
)

# Khởi tạo Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), 
    train_dataset = train_dataset,
    eval_dataset = val_dataset,         
    args = training_args,
    
    # --- THÊM HÀM METRIC VÀO TRAINER ---
    compute_metrics = compute_metrics,                       
    preprocess_logits_for_metrics = preprocess_logits_for_metrics, 
    
    # Điều chỉnh patience: Dừng sau 5 lần đánh giá (tương đương 250 steps) nếu không tăng
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Bắt đầu huấn luyện
trainer.train()

# Lưu mô hình
model.save_pretrained("/kaggle/working/qwen2_vl_2b_finetuned")
tokenizer.save_pretrained("/kaggle/working/qwen2_vl_2b_finetuned")
print("Đã lưu mô hình thành công tại /kaggle/working/qwen2_vl_2b_finetuned")

Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,075 | Num Epochs = 5 | Total steps = 1,300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Vqa Accuracy
50,0.635700,0.613811,0.000000
100,0.556300,0.513449,0.003663
150,0.429500,0.471614,0.000000
200,0.382000,0.437746,0.003663
250,0.436400,0.419866,0.003663
300,0.352300,0.419214,0.007326
350,0.366000,0.412303,0.000000
400,0.354600,0.399639,0.000000
450,0.340900,0.399124,0.003663
500,0.345100,0.391533,0.003663


Unsloth: Not an error, but Qwen2VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Đã lưu mô hình thành công tại /kaggle/working/qwen2_vl_2b_finetuned
